In [44]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
import scipy.signal as signal
from scipy.signal import butter, freqz, lfilter, bilinear
from ipywidgets import interact

def buck_boost_sim(D=0.3, Fsw=1000, Vi=30, L=50e-3, R=100, C=10e-6, Ciclos=5, Eje=1.5, Transitorio=False):
    print(f"L = {L:.5g} H")
    print(f"C = {C:.5g} F")

    Tsw=1/Fsw
    Vo=(Vi*D)/(1-D)
    Io=Vo/R
    Il_dc=Io/(1-D)
    delta_Il=(Vi*D*Tsw)/L
    Il_max=Il_dc+(delta_Il/2)
    Il_min=Il_dc-(delta_Il/2)

    if Il_min <= 0:
        print('Modo discontinuo, modifique los parámetros')
        return
    
    puntos=int(1e4)
    t=np.linspace(0, 1, puntos, endpoint=False)
    
    indice = int(D * puntos)
    xd = np.zeros(puntos)
    xs = np.zeros(puntos)
    xd[indice:] = np.linspace(Il_max, Il_min, puntos - indice)
    xs[:indice] = np.linspace(Il_min, Il_max, indice)
    
    xxd = np.tile(xd, 100)
    xxs = np.tile(xs, 100)
    
    Wn_norm = (Fsw / (Fsw * puntos)) * 2  # Normalizado a Nyquist
    
    num=R
    den=[R*C,1]
    b, a=bilinear(num,den,Fsw*puntos) 
    
    vo_filtrada = lfilter(b, a, xxd)
    
    tiempo_total = np.linspace(0,100*Tsw, len(xxd))
    
    if Transitorio:
        tiempo_mostrar = tiempo_total[:puntos * Ciclos]
        vo_filtrada_mostrar = vo_filtrada[:len(tiempo_mostrar)]
        xxs_mostrar = xxs[:len(tiempo_mostrar)]
        xxd_mostrar = xxd[:len(tiempo_mostrar)]
    else:
        tiempo_mostrar = tiempo_total[-puntos * Ciclos:]
        vo_filtrada_mostrar = vo_filtrada[-len(tiempo_mostrar):]
        xxs_mostrar = xxs[-len(tiempo_mostrar):]
        xxd_mostrar = xxd[-len(tiempo_mostrar):]


    vo_freq=np.fft.fftshift(abs(np.fft.fft(vo_filtrada))/len(vo_filtrada))
    x_four=(np.arange(len(vo_filtrada))/len(vo_filtrada)-0.5)*Fsw*puntos

    w, h = signal.freqz(b, a, worN=len(vo_filtrada), fs=Fsw * puntos)
    h = np.abs(h) / np.max(np.abs(h)) * np.max(vo_freq)  # Normalización de la envolvente
    w_reflect = -w  # Reflexión para graficar simétricamente
    
    plt.figure(figsize=(12, 12))
    plt.subplot(3, 1, 1)
    plt.plot(tiempo_mostrar, vo_filtrada_mostrar, 'r')
    plt.ylim([-1/10*min(vo_filtrada_mostrar) + min(vo_filtrada_mostrar),1/12*min(vo_filtrada_mostrar) + max(vo_filtrada_mostrar)])
    plt.title('Voltaje de salida filtrado')
    plt.xlabel('Tiempo (s)')
    plt.ylabel('Voltaje (V)')
    plt.grid()
    
    plt.subplot(3, 1, 2)
    plt.plot(tiempo_mostrar, xxd_mostrar, label='Corriente Diodo')
    plt.plot(tiempo_mostrar, xxs_mostrar, '--', label='Corriente Transistor')
    plt.xlabel('Tiempo (s)')
    plt.ylabel('Corriente (A)')
    plt.legend(loc='upper right')
    plt.grid()
    
    D_transf = np.linspace(0, 0.99, 100)
    M = -D_transf / (1 - D_transf)
    plt.subplot(3, 1, 3)
    plt.plot(D_transf, M, 'b')
    plt.plot(D, -D / (1 - D), 'ro')
    plt.xlabel('Ciclo de trabajo D')
    plt.ylabel('M')
    plt.title('Curva de transferencia')
    plt.grid()

    plt.subplot(3, 1, 3)
    plt.plot(x_four, vo_freq, 'b', label="FFT de $V_o$")
    plt.plot(w, h, 'r-.', label="Envolvente del filtro")
    plt.plot(w_reflect, h, 'r-.')  # Reflexión simétrica
    plt.xlabel('Frecuencia (Hz)')
    plt.ylabel('Magnitud')
    plt.title('Espectro de frecuencias y envolvente del filtro')
    plt.xlim([-Eje * Fsw, Eje * Fsw])  # Rango de frecuencias mostrado
    plt.ylim([0, max(vo_freq)+1])
    plt.legend()
    plt.grid()


    plt.tight_layout()
    plt.show()


    
#interact(buck_boost_sim, D=(0.05, 0.95, 0.01), Fsw=(500, 5000, 500), Vi=(10, 50, 5), L=(10e-3, 100e-3, 5e-3), R=(50, 200, 10), C=(1e-6, 50e-6, 1e-6), Ciclos=(1, 10, 1), Transitorio=[False, True])
interact(buck_boost_sim, 
    D=widgets.FloatSlider(min=0.05, max=0.95, step=0.01, value=0.3, description="D", format=".2e"),
    Fsw=widgets.FloatSlider(min=500, max=5000, step=500, value=1000, description="Fsw", format=".2e"),
    Vi=widgets.FloatSlider(min=10, max=50, step=5, value=30, description="Vi", format=".2e"),
    L=widgets.FloatSlider(min=10e-3, max=100e-3, step=5e-3, value=50e-3, description="L", format=".2e"),
    R=widgets.FloatSlider(min=50, max=200, step=10, value=100, description="R", format=".2e"),
    C=widgets.FloatSlider(min=1e-6, max=50e-6, step=1e-6, value=10e-6, description="C", format=".2e"),
    Ciclos=widgets.IntSlider(min=1, max=10, step=1, value=5, description="Ciclos"),
    Eje=widgets.FloatSlider(min=0.5, max=10, step=1, description="Eje de frecuencias"),
    Transitorio=widgets.Checkbox(value=False, description="Transitorio"),
)



interactive(children=(FloatSlider(value=0.3, description='D', max=0.95, min=0.05, step=0.01), FloatSlider(valu…

<function __main__.buck_boost_sim(D=0.3, Fsw=1000, Vi=30, L=0.05, R=100, C=1e-05, Ciclos=5, Eje=1.5, Transitorio=False)>